> Experiments with [model.signal](../trapallada/model/signal.py)  module

In [1]:
import pathlib

import yaml

import jax.random as jr

import trapallada.data
import trapallada.model.nn
import trapallada.util
import trapallada.core

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Directory containing the data
- `data/mini_rates` was built running `just pack_obs_test mini_rates`

In [3]:
# input_directory = pathlib.Path.cwd().parent / 'samples' / 'data' / 'mini_rate'
# input_directory = pathlib.Path.cwd().parent / 'data' / 'rate_backv'
input_directory = pathlib.Path.cwd().parent / 'data' / 'mini_rates'
assert input_directory.exists()
input_directory

PosixPath('/home/manu/Sync/git/trapallada/data/mini_rates')

Data is stored in three separate files:

- signals, along with their corresponding...
- masks, both in the form of `Array`s, and
- metadata as a *Pandas* `DataFrame`

In [4]:
signals_file = input_directory / 'signals.pt'
assert signals_file.exists()

masks_file = input_directory / 'masks.pt'
assert masks_file.exists()

metadata_file = input_directory / 'metadata.parquet'
assert metadata_file.exists

It it turned into a *PyTorch* `Dataset`

In [5]:
ds = trapallada.data.SignalsDataset(signals_file, metadata_file, masks_file)
len(ds), ds[0][1].shape

(236, torch.Size([5, 225]))

A parameters file that specifies an experiment

In [6]:
parameters_file = pathlib.Path.cwd().parent / 'parameters.yaml'
# parameters_file = pathlib.Path.cwd().parent / 'samples' / 'fft_vanilla_mini.yaml'
assert parameters_file.exists()

It is read into a `dict`

In [7]:
with parameters_file.open() as f:
    parameters = yaml.safe_load(f)

In [8]:
%pycat {parameters_file}

PRNG seeds:
  model initialization:
    decoder: 43
    encoder: 42
  sampling:
    training set: 14
    validation set: 15
  training dataloader: 43
  training-validation splitting: 42
loss:
  x_sd: 0.15 # <---- FFT
model: &model
  # channels: [12, 12, 12]
  channels:
  - 8
  - 16
  - 32
  dim_z: 40
  kernel_sizes: # <---- FFT (due to redundancy)
  - 64
  - 32
  - 16
  # stride: 2
  stride:
  - 2
  - 2
  - 2
data:
    fft: false # <---- FFT
training:
  batch_size: 32
  dl_rng_key: 43
  learning_rate: 3.0e-05
  n_epochs: 2
  
  # ratio of the dataset to use for validation
  train_test_split:
    test_size: 0.2
  
  # whether or not to ignore non "good time intervals" (GTIs) in the loss function; it only
  # makes sense when working with time-domain signals, and will be ignored for frequency-domain ones
  mask_non_gti_intervals: true
VAE:
  class: VAE
  encoder:
    class: Encoder
    parameters:
      << : [*model]
  decoder:
    class: Decoder # <---- FFT (NonNegativeDecoder/NonNegati

# Data

`DataLoaders` are derived from the `Dataset`

In [9]:
train_dl, validation_dl = trapallada.data.dataloaders_from_dataset(
    ds,
    parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    parameters['training']['batch_size'], parameters['PRNG seeds']['training dataloader'])


How many samples per batch?

In [10]:
batch = next(iter(train_dl))
len(batch)

3

The *index component* in the first batch

In [11]:
batch[0]

tensor([113,  78, 115, 126, 143,  83,  68, 116,  37, 167, 203, 154, 166,  46,
        135, 207,  50,  64, 104, 216,  87,  72, 205,  51,  88,  79,  80, 217,
         47, 163, 175, 151])

The *x component* in the 1st element of the `Dataset`

In [12]:
signals_len = ds[0][1].shape
signals_len

torch.Size([5, 225])

What are the dimensions of every input?

In [13]:
tuple(ds[0][1].shape)

(5, 225)

Both *encoder* and *decoder* also require the *input size* as a parameter: it's inferred from the `Dataset` and included in the corresponding `dict`s

In [14]:
input_size = list(ds[0][1].shape)
for m in ['encoder', 'decoder']:
        parameters['VAE'][m]['parameters']['input_size'] = input_size

# Encoder

In [15]:
x = batch[1][0].numpy()
encoder = trapallada.model.nn.Encoder(
    **parameters['VAE']['encoder']['parameters'],
    key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['encoder']))
encoder.debug_call(x)
print(encoder)

after equinox.nn._conv: (8, 81)
after jax._src.nn.functions: (8, 81)
after equinox.nn._normalisation: (8, 81)
after equinox.nn._conv: (16, 25)
after jax._src.nn.functions: (16, 25)
after equinox.nn._normalisation: (16, 25)
after equinox.nn._conv: (32, 5)
after jax._src.nn.functions: (32, 5)
after equinox.nn._normalisation: (32, 5)
after jax._src.numpy.lax_numpy: (160,)
after equinox.nn._linear: (80,)
after trapallada.model.nn: (80,)
Encoder(
  layers=[
    Conv1d(
      num_spatial_dims=1,
      weight=f32[8,5,64],
      bias=f32[8,1],
      in_channels=5,
      out_channels=8,
      kernel_size=(64,),
      stride=(2,),
      padding=((0, 0),),
      dilation=(1,),
      groups=1,
      use_bias=True
    ),
    <wrapped function relu>,
    LayerNorm(
      shape=(8, 81),
      eps=1e-05,
      use_weight=True,
      use_bias=True,
      weight=f32[8,81],
      bias=f32[8,81]
    ),
    Conv1d(
      num_spatial_dims=1,
      weight=f32[16,8,32],
      bias=f32[16,1],
      in_channels

In [16]:
x.shape

(5, 225)

In [17]:
z = encoder(x)
z.shape

(80,)

# Decoder

In [18]:
z_mean = jr.normal(jr.PRNGKey(42), (parameters['model']['dim_z'],))
decoder = trapallada.model.nn.Decoder(
    **(parameters['VAE']['decoder']['parameters'] | {'layers_spatial_sizes': encoder.layers_spatial_sizes}),
    key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['decoder']))
decoder.debug_call(z_mean)

after equinox.nn._linear: (160,)
after jax._src.nn.functions: (160,)
after trapallada.model.nn: (32, 5)
after equinox.nn._conv: (16, 25)
after jax._src.nn.functions: (16, 25)
after equinox.nn._normalisation: (16, 25)
after equinox.nn._conv: (8, 81)
after jax._src.nn.functions: (8, 81)
after equinox.nn._normalisation: (8, 81)
after equinox.nn._conv: (5, 225)


In [19]:
vae = trapallada.model.nn.VAE(encoder, decoder)
x_pred, z_mean, z_sd = vae(x, jr.PRNGKey(42))
x_pred.shape, z_mean.shape, z_sd.shape

((5, 225), (40,), (40,))

Training

In [20]:
model, results_df = trapallada.core.fit_vae_on_signals(train_dl, validation_dl, parameters)

trapallada.core - INFO - model has 53,733 parameters


    training                         validation                        
        loss loglikelihood        kl       loss loglikelihood        kl
0  93.328342    -93.261697  0.066645  93.327213    -93.259712  0.067501
    training                         validation                        
        loss loglikelihood        kl       loss loglikelihood        kl
1  92.365348    -92.298684  0.066663  92.761631    -92.694126  0.067504


# Conditional VAE

See [A deep dive into conditional variational autoencoders](https://beckham.nz/2023/04/27/conditional-vaes.html) (also [1](https://towardsdatascience.com/conditional-variational-autoencoders-with-learnable-conditional-embeddings-e22ee5359a2a) and [2](https://towardsdatascience.com/understanding-conditional-variational-autoencoders-cd62b4f57bf8))

In [21]:
cond_decoder = trapallada.model.nn.ConditionalDecoder(
    **(parameters['VAE']['decoder']['parameters'] | {'input_size': [1, input_size[1]]}| {'layers_spatial_sizes': encoder.layers_spatial_sizes}),
    key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['decoder']))

In [22]:
x_pred = cond_decoder(z_mean, x[-1:])
x_pred.shape

(1, 225)

Notice the **encoder is the same** as before.

In [23]:
cvae = trapallada.model.nn.CVAE(encoder, cond_decoder)
x_pred, z_mean, z_sd = cvae(x, jr.PRNGKey(42))
x_pred.shape, z_mean.shape, z_sd.shape

((1, 225), (40,), (40,))

In [24]:
import pandas as pd
pd.Timedelta(1.46, 'seconds')

Timedelta('0 days 00:00:01.460000')

# GMEncoder

In [25]:
encoder_parameters = parameters['VAE']['encoder']['parameters'].copy()
del encoder_parameters['dim_z']
encoder_parameters['dim_y'] = 3
encoder_parameters

{'channels': [8, 16, 32],
 'kernel_sizes': [64, 32, 16],
 'stride': [2, 2, 2],
 'input_size': [5, 225],
 'dim_y': 3}

In [26]:
# x = batch[1][0].numpy()
# encoder = trapallada.model.nn.GMEncoder(
#     **encoder_parameters,
#     key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['encoder']))
# # encoder.debug_call(x)
# # print(encoder)
# y = encoder(x)
# # assert y.sum() == 1
# y

In [27]:
# encoder.latent_given_class_and_data

In [28]:
# import equinox as eqx
# encoder.class_given_data
# # eqx.tree_at(encoder.class_given_data, 2)
# # eqx.filter(encoder, (True, [True]))
# eqx.tree_at(lambda m: m.layers, encoder, replace=encoder.layers[:-2])

In [29]:
# encoder.latent_given_class_and_data